# Convert one short-story result JSON to CSV

This notebook loads one single-document result file, finds its source story, aligns each predicted action to a sentence, and writes a two-column CSV containing `Sentence` and `NL2P`. Each `NL2P` value is a list of action records with both `verb` and `arguments`.

In [1]:
import json
import re
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root():
    """Find the repository root whether Jupyter starts there or in extra/."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "data/short_stories").is_dir() and (candidate / "results").is_dir():
            return candidate
    raise RuntimeError("Could not find the llm-action-extraction project root.")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from evaluation_helpers import nlp

RESULTS_DIR = PROJECT_ROOT / "results/short_stories/nl2p_1/gpt-5"
STORIES_PATH = PROJECT_ROOT / "data/short_stories/mulab_short_stories/selected_stories.json"

# Input: change only this filename to convert another single-document result.
INPUT_FILE = "chicken-licken.json"
OUTPUT_CSV = PROJECT_ROOT / "extra" / f"{Path(INPUT_FILE).stem}_nl2p.csv"

## Alignment strategy

The alignment follows the intended temporal-order behavior of `diagnosis/read_results_into_table.ipynb`:

- Actions stay in their original order.
- The leading verb word is matched case-insensitively as a complete word. For a multiword verb, every remaining verb word must occur in order within the next five sentence-word tokens; intervening words are allowed.
- Every listed argument must also occur as an exact, case-insensitive phrase in the same sentence. Actions with no arguments are matched by the verb alone.
- Matching starts at the previously matched sentence and strictly moves forward or stays on that sentence. Multiple actions may align to one sentence.
- Earlier sentences are never searched. If no match exists at or after the cursor, the action is collected in a final `Unmatched` row after all source sentences.
- Unmatched actions do not move the sentence cursor, so later actions can still match in temporal order.
- Source text is selected by its SHA-256 hash from `selected_stories.json`, then split into sentences with the repository's spaCy pipeline.

In [2]:
WORD_PATTERN = re.compile(r"[A-Za-z0-9]+(?:[-'’][A-Za-z0-9]+)*")
VERB_CONTEXT_WINDOW = 5


def word_tokens(text):
    """Return case-folded word tokens used by verb-window matching."""
    return [match.group(0).casefold() for match in WORD_PATTERN.finditer(str(text))]


def verb_occurs(verb, sentence, context_window=VERB_CONTEXT_WINDOW):
    """Match a verb's leading word and its ordered tail within a forward window."""
    verb_words = word_tokens(verb)
    sentence_words = word_tokens(sentence)
    if not verb_words:
        return False

    leading_word, *remaining_words = verb_words
    for leading_index, sentence_word in enumerate(sentence_words):
        if sentence_word != leading_word:
            continue
        if not remaining_words:
            return True

        window = sentence_words[
            leading_index + 1 : leading_index + 1 + context_window
        ]
        window_index = 0
        for remaining_word in remaining_words:
            try:
                matched_offset = window.index(remaining_word, window_index)
            except ValueError:
                break
            window_index = matched_offset + 1
        else:
            return True

    return False


def argument_occurs(argument, sentence):
    """Return whether a complete argument phrase occurs in a sentence."""
    words = str(argument).split()
    if not words:
        return False

    phrase_pattern = r"\s+".join(re.escape(word) for word in words)
    pattern = rf"(?<!\w){phrase_pattern}(?!\w)"
    return re.search(pattern, sentence, flags=re.IGNORECASE) is not None


def action_occurs(action, sentence):
    """Return whether the verb and every argument occur in one sentence."""
    return verb_occurs(action["verb"], sentence) and all(
        argument_occurs(argument, sentence)
        for argument in action["arguments"]
    )


def load_result(filename):
    """Load one result object from RESULTS_DIR using its filename."""
    input_path = Path(filename)
    if not input_path.is_absolute():
        input_path = RESULTS_DIR / input_path

    with input_path.open(encoding="utf-8") as file:
        result = json.load(file)

    if not isinstance(result, dict) or not isinstance(result.get("actions"), list):
        raise ValueError(
            f"{input_path} is not a single-document result with an actions list. "
            "Choose one of the per-story JSON files rather than results.json."
        )
    return result


def load_source_story(result):
    """Find the unique source story associated with a result object."""
    with STORIES_PATH.open(encoding="utf-8") as file:
        stories = json.load(file)["stories"]

    content_hash = result.get("content_sha256")
    matches = [story for story in stories if story.get("content_sha256") == content_hash]
    if len(matches) != 1:
        dataset_title = result.get("dataset_title")
        matches = [story for story in stories if story.get("dataset_title") == dataset_title]
    if len(matches) != 1:
        raise ValueError(
            "Expected exactly one source story for "
            f"{result.get('dataset_title')!r}; found {len(matches)}."
        )
    return matches[0]


def split_sentences(content):
    """Split source content into non-empty sentences with spaCy."""
    sentences = [sentence.text.strip() for sentence in nlp(content).sents if sentence.text.strip()]
    if not sentences:
        raise ValueError("The source story did not contain any sentences.")
    return sentences


def align_actions_to_sentences(sentences, actions):
    """Align actions monotonically and return sentence groups plus unmatched actions."""
    aligned = [[] for _ in sentences]
    unmatched = []
    matched_sentence_indices = []
    sentence_index = 0

    for action_index, action in enumerate(actions):
        if not isinstance(action, dict) or not str(action.get("verb", "")).strip():
            raise ValueError(f"Action {action_index} has no valid verb: {action!r}")

        verb = action["verb"]
        arguments = action.get("arguments", [])
        if not isinstance(arguments, list) or not all(
            isinstance(argument, str) and argument.strip()
            for argument in arguments
        ):
            raise ValueError(f"Action {action_index} has invalid arguments: {arguments!r}")
        normalized_action = {"verb": verb, "arguments": arguments}

        match_index = next(
            (
                index
                for index in range(sentence_index, len(sentences))
                if action_occurs(normalized_action, sentences[index])
            ),
            None,
        )
        if match_index is None:
            unmatched.append(normalized_action)
            continue

        if match_index < sentence_index:
            raise AssertionError(
                f"Temporal order violation: sentence {match_index} follows cursor "
                f"{sentence_index} for action {action_index}."
            )
        aligned[match_index].append(normalized_action)
        matched_sentence_indices.append(match_index)
        sentence_index = match_index

    if any(
        earlier > later
        for earlier, later in zip(
            matched_sentence_indices, matched_sentence_indices[1:]
        )
    ):
        raise AssertionError("Matched sentence indices are not monotonically ordered.")

    return aligned, unmatched


def result_json_to_dataframe(filename):
    """Build sentence/NL2P rows followed by one optional unmatched row."""
    result = load_result(filename)
    story = load_source_story(result)
    sentences = split_sentences(story["content"])
    aligned_actions, unmatched_actions = align_actions_to_sentences(
        sentences, result["actions"]
    )
    rows = [
        {"Sentence": sentence, "NL2P": actions}
        for sentence, actions in zip(sentences, aligned_actions)
    ]
    if unmatched_actions:
        rows.append({"Sentence": "Unmatched", "NL2P": unmatched_actions})
    return pd.DataFrame(rows, columns=["Sentence", "NL2P"])


def result_json_to_csv(filename, output_csv=None):
    """Align one result filename and save its Sentence/NL2P CSV."""
    dataframe = result_json_to_dataframe(filename)

    output_path = (
        Path(output_csv)
        if output_csv is not None
        else PROJECT_ROOT / "extra" / f"{Path(filename).stem}_nl2p.csv"
    )
    output_path.parent.mkdir(parents=True, exist_ok=True)
    csv_dataframe = dataframe.copy()
    csv_dataframe["NL2P"] = csv_dataframe["NL2P"].map(
        lambda actions: json.dumps(actions, ensure_ascii=False)
    )
    csv_dataframe.to_csv(output_path, index=False)
    return dataframe, output_path


def display_wrapped_dataframe(dataframe):
    """Display fully wrapped sentence text in Jupyter."""
    with pd.option_context("display.max_colwidth", None):
        display(
            dataframe.style.set_properties(
                **{
                    "white-space": "normal",
                    "overflow-wrap": "anywhere",
                    "vertical-align": "top",
                    "text-align": "left",
                }
            )
        )

In [3]:
aligned_dataframe, saved_csv = result_json_to_csv(INPUT_FILE, OUTPUT_CSV)
unmatched_mask = aligned_dataframe["Sentence"].eq("Unmatched")
unmatched_count = (
    len(aligned_dataframe.loc[unmatched_mask, "NL2P"].iloc[0])
    if unmatched_mask.any()
    else 0
)
action_count = sum(len(actions) for actions in aligned_dataframe["NL2P"])
sentence_row_count = len(aligned_dataframe) - int(unmatched_mask.any())
print(
    f"Saved {sentence_row_count} sentence rows containing "
    f"{action_count - unmatched_count} aligned actions and "
    f"{unmatched_count} unmatched actions to {saved_csv}"
)
display_wrapped_dataframe(aligned_dataframe)

Saved 38 sentence rows containing 21 aligned actions and 1 unmatched actions to /home/char37/Projects/MPhil/llm-action-extraction/extra/chicken-licken_nl2p.csv


,Sentence,NL2P
0,"As Chicken-licken was going one day to the wood, whack!","[{'verb': 'was going', 'arguments': ['Chicken-licken']}]"
1,an acorn fell from a tree on to his head.,"[{'verb': 'fell', 'arguments': ['an acorn']}]"
2,"""Gracious goodness me!"" said Chicken-licken, ""the sky must have fallen; I must go and tell the King.""",[]
3,"So Chicken-licken turned back, and met Hen-len.","[{'verb': 'turned back', 'arguments': ['Chicken-licken']}, {'verb': 'met', 'arguments': ['Hen-len']}, {'verb': 'turned back', 'arguments': ['Hen-len']}]"
4,"""Well, Hen-len, where are you going?"" said he.",[]
5,"""I'm going to the wood,"" said she.",[]
6,"""Oh, Hen-len, don't go!"" said he, ""for as I was going the sky fell on to my head, and I'm going to tell the King.""",[]
7,"So Hen-len turned back with Chicken-licken, and met Cock-lock.","[{'verb': 'met', 'arguments': ['Cock-lock']}, {'verb': 'turned back', 'arguments': ['Cock-lock']}]"
8,"""I'm going to the wood,"" said he.",[]
9,"Then Hen-len said: ""Oh Cock-lock, don't go, for I was going, and I met Chicken-licken, and Chicken-licken had been at the wood, and the sky had fallen on to his head, and we are going to tell the King.""",[]
